# Lesson 05 — Histogram Backprojection

## Why This Lesson
Backprojection answers: "for every pixel in this image, how likely is it to match my target object's color?" This is the math behind color-based object tracking (CAMShift).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread('sample.jpg')
# Define ROI — the region whose color you want to find elsewhere
roi = img[80:180, 80:180]   # adjust to a colorful region in your image

hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

# Build H-S histogram of ROI
roi_hist = cv2.calcHist([hsv_roi], [0, 1], None, [180, 256], [0, 180, 0, 256])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# Project back: each pixel = probability it matches ROI color
backproj = cv2.calcBackProject([hsv_img], [0, 1], roi_hist, [0, 180, 0, 256], 1)

# Clean up with disc filter
disc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
cv2.filter2D(backproj, -1, disc, backproj)

_, mask   = cv2.threshold(backproj, 50, 255, cv2.THRESH_BINARY)
result    = cv2.bitwise_and(img, img, mask=mask)

# Draw ROI on original
vis = img.copy()
cv2.rectangle(vis, (80,80), (180,180), (0,255,0), 2)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(cv2.cvtColor(vis,     cv2.COLOR_BGR2RGB)); axes[0].set_title('Image + ROI (green box)'); axes[0].axis('off')
axes[1].imshow(backproj, cmap='hot');                     axes[1].set_title('Backprojection probability map'); axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(result,  cv2.COLOR_BGR2RGB)); axes[2].set_title('Regions matching ROI color'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## Key Takeaway
High backprojection value = "this pixel's color looks like my target." It's a probability map. This is exactly how CAMShift tracking initializes.